In [1]:
!pip install -U -q git+https://github.com/huggingface/trl.git bitsandbytes peft qwen-vl-utils trackio
# Tested with trl==0.22.0.dev0, bitsandbytes==0.47.0, peft==0.17.1, qwen-vl-utils==0.0.11, trackio==0.2.8

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
system_message = """
      You are a forensic video analyst specializing in sexual content detection. You describe what you see in tremendous detail. Also, you perform structured analysis of visual cues that signal sexual content. mention each and every small detail (whether it be redundant or not useful)

      For EACH distinct scene in the video, analyze these 5 dimensions and score each 0-5:

      1. BODY LANGUAGE & POSTURE
      2. CLOTHING & APPEARANCE
      3. SETTING & PROPS
      4. PHYSICAL INTERACTION (if multiple people)
      5. TEXT & AUDIO CUES (if visible/audible)
      6. EXPLICIT CONTENT ANALYSIS (NUDITY/PENETRATION)

      Your analysis should include each of these parameters"""

In [3]:
def format_data(sample):
    return {
    #   "videos": [sample["video"]],
      "messages": [

          {
              "role": "system",
              "content": [
                  {
                      "type": "text",
                      "text": system_message
                  }
              ],
          },
          {
              "role": "user",
              "content": [
                  {
                      "type": "video",
                      "url": sample["url"],
                  },
                  {
                      "type": "text",
                      "text": "What is in the video?",
                  }
              ],
          },
          {
              "role": "assistant",
              "content": [
                  {
                      "type": "text",
                      "text": sample["annotation"]
                  }
              ],
          },
      ]
      }

In [4]:
from datasets import load_dataset

dataset_id = "1rsh/mtp-ii-reels"
train_dataset = load_dataset(dataset_id, split='train')

Resolving data files:   0%|          | 0/268 [00:00<?, ?it/s]

In [5]:
train_dataset = train_dataset.shuffle(seed=4)

In [6]:
train_dataset = train_dataset.select(range(50))  # Select the first 50 samples for training

In [7]:
def to_direct_download(url: str) -> str:
    file_id = url.split("/d/")[1].split("/")[0]
    return f"https://drive.google.com/uc?export=download&id={file_id}"

train_dataset = train_dataset.map(lambda x: {**x, "url": to_direct_download(x['shareable_link'])})

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [8]:
train_dataset = [format_data(sample) for sample in train_dataset]

In [9]:
train_dataset[0]

{'messages': [{'role': 'system',
   'content': [{'type': 'text',
     'text': '\n      You are a forensic video analyst specializing in sexual content detection. You describe what you see in tremendous detail. Also, you perform structured analysis of visual cues that signal sexual content. mention each and every small detail (whether it be redundant or not useful)\n\n      For EACH distinct scene in the video, analyze these 5 dimensions and score each 0-5:\n\n      1. BODY LANGUAGE & POSTURE\n      2. CLOTHING & APPEARANCE\n      3. SETTING & PROPS\n      4. PHYSICAL INTERACTION (if multiple people)\n      5. TEXT & AUDIO CUES (if visible/audible)\n      6. EXPLICIT CONTENT ANALYSIS (NUDITY/PENETRATION)\n\n      Your analysis should include each of these parameters'}]},
  {'role': 'user',
   'content': [{'type': 'video',
     'url': 'https://drive.google.com/uc?export=download&id=1xcPI__8C43aqj0uhjnHiatSREVzR6mVM'},
    {'type': 'text', 'text': 'What is in the video?'}]},
  {'role': 'a

In [10]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, Qwen2_5_VLProcessor

model_id = "huihui-ai/Qwen2.5-VL-3B-Instruct-abliterated"

In [11]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

processor = Qwen2_5_VLProcessor.from_pretrained(model_id)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

In [12]:
from qwen_vl_utils import process_vision_info

def generate_text_from_sample(model, processor, sample, max_new_tokens=1024, device="mps"):
    # Prepare the text input by applying the chat template
    text_input = processor.apply_chat_template(
        sample['messages'][1:2],  # Use the sample without the system message
        tokenize=False,
        add_generation_prompt=True
    )

    # Process the visual input from the sample
    _, video_inputs = process_vision_info(sample['messages'])

    # Prepare the inputs for the model
    model_inputs = processor(
        text=[text_input],
        videos=video_inputs,
        return_tensors="pt",
    ).to(device)  # Move inputs to the specified device

    # Generate text with the model
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)

    # Trim the generated ids to remove the input ids
    trimmed_generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decode the output text
    output_text = processor.batch_decode(
        trimmed_generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )

    return output_text[0]  # Return the first decoded output text

In [14]:
# Example of how to call the method with sample:
output = generate_text_from_sample(model, processor, train_dataset[0])
output

ValueError: image, image_url or video should in content.

In [13]:
from transformers import BitsAndBytesConfig

# BitsAndBytesConfig int-4 config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [14]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config
)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

/Users/irsh/Documents/workdir/academic/mtp/agentic/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/Users/irsh/Documents/workdir/academic/mtp/agentic/.venv/lib/python3.11/site-packages/bitsandbytes/backends/default/ops.py:178: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [22]:
from peft import LoraConfig

# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=4,
    bias="none",
    target_modules=["down_proj", "up_proj"],  # Target both down_proj and up_proj for better performance
    task_type="CAUSAL_LM",
)

In [23]:
from trl import SFTConfig

# Configure training arguments
training_args = SFTConfig(
    output_dir="qwen2.5-3b-instruct-implicit-reels",  # Directory to save the model
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size for training
    # per_device_eval_batch_size=4,  # Batch size for evaluation
    gradient_accumulation_steps=8,  # Steps to accumulate gradients
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Options for gradient checkpointing
    max_length=None,
    # Optimizer and scheduler settings
    optim="adamw_torch_fused",  # Optimizer type
    learning_rate=2e-4,  # Learning rate for training
    # Logging and evaluation
    logging_steps=10,  # Steps interval for logging
    # eval_steps=10,  # Steps interval for evaluation
    # eval_strategy="steps",  # Strategy for evaluation
    save_strategy="steps",  # Strategy for saving the model
    save_steps=20,  # Steps interval for saving
    # Mixed precision and gradient settings
    bf16=True,  # Use bfloat16 precision
    max_grad_norm=0.3,  # Maximum norm for gradient clipping
    warmup_ratio=0.03,  # Ratio of total steps for warmup
    # Hub and reporting
    push_to_hub=True,  # Whether to push model to Hugging Face Hub
    report_to="trackio",  # Reporting tool for tracking metrics
    trackio_space_id="1rsh/qwen2.5-3b-instruct-implicit-reels"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [24]:
import trackio

trackio.init(
    project="qwen2.5-3b-instruct-implicit-reels",
    name="qwen2.5-3b-instruct-implicit-reels",
    config=training_args,
    space_id=training_args.output_dir + "-trackio"
)

* Run finished. Uploading logs to the remote Trackio server (please wait...)
* Trackio project initialized: qwen2.5-3b-instruct-implicit-reels
* Trackio metrics will be synced to Hugging Face Bucket: https://huggingface.co/buckets/1rsh/qwen2.5-3b-instruct-implicit-reels-trackio-bucket
* Found existing space: https://huggingface.co/spaces/1rsh/qwen2.5-3b-instruct-implicit-reels-trackio


* Created new run: qwen2.5-3b-instruct-implicit-reels


In [25]:
processor = Qwen2_5_VLProcessor.from_pretrained(model_id)

In [ ]:
from trl import SFTTrainer
from datasets import Dataset

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=Dataset.from_list(train_dataset),
    peft_config=peft_config,
    processing_class=processor,
)

/Users/irsh/Documents/workdir/academic/mtp/agentic/.venv/lib/python3.11/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/Users/irsh/Documents/workdir/academic/mtp/agentic/.venv/lib/python3.11/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (146385 > 131072). Running this sequence through the model will result in indexing errors


In [ ]:
trainer.train()

/Users/irsh/Documents/workdir/academic/mtp/agentic/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: Invalid buffer size: 63.41 GiB

In [ ]:
trainer.save_model(training_args.output_dir)

In [ ]:
output = generate_text_from_sample(model, processor, train_dataset[5])
output

In [ ]:
train_dataset[5]